# ML08 · 训练的艺术

把一个能跑的神经网络，调教成「在没看过的数据上也表现好」的模型。

## 学习目标
- 理解小批量（mini-batch）与数据加载器（DataLoader）的概念，知道为什么不一次喂全部数据。
- 分辨优化器（optimizer）SGD 与 Adam 的差异，知道何时选哪一个。
- 会做训练/验证切分（train/validation split），并用它侦测过拟合（overfitting）。
- 学会两种对抗过拟合的工具：丢弃法（dropout）与提早停止（early stopping）。
- 能判读学习曲线（learning curve），从曲线形状诊断训练状况并决定下一步。

In [ ]:
# 概念：统一导入与画图设置，后面各范例共用；固定乱数种子让结果可重现
import numpy as np
import matplotlib.pyplot as plt

# 让 matplotlib 在 notebook 内直接显示图
%matplotlib inline

# 设置中文字体，避免图上中文变成方框（找不到也不影响数值与英文标签）
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False   # 负号正常显示

print("环境就绪，numpy 版本:", np.__version__)

## 小批量与数据加载

小批量（mini-batch）是指每次参数更新只看「一小撮」样本，而不是全部数据。一次完整看过所有数据叫一个世代（epoch）；一个 epoch 内会切成很多小批量依序更新。

为什么这样做：更新更频繁、内存更省，而且小批量带来的梯度杂讯反而有助于跳出不好的点。批量大小（batch size）设大时更新次数少、每步较稳；设小时更新次数多、杂讯较大。数据加载器（DataLoader）就是「自动分批与洗牌（shuffle）」的搬运工。

形状：一个 epoch 的更新次数约为 样本数 / batch size（无条件进位）。

In [ ]:
# 概念：手写一个迷你分批器（mini-batch iterator），每个 epoch 先洗牌再切成固定大小的批量
rng = np.random.default_rng(0)   # 固定种子，方便对照

# 造一批假样本：每栋建筑有 楼高(公尺) 与 面积(平方公尺) 两个特征
n_samples = 20
features = rng.uniform(low=[6, 50], high=[60, 240], size=(n_samples, 2))

def iterate_minibatches(data, batch_size, shuffle=True):
    n = len(data)
    indices = np.arange(n)
    if shuffle:
        rng.shuffle(indices)              # 洗牌：打散样本顺序，避免排序影响更新
    for start in range(0, n, batch_size):
        batch_idx = indices[start:start + batch_size]   # 取这一批的列索引
        yield data[batch_idx]             # 用 yield 逐批吐出，省内存

# 比较 batch size 设大与设小时，一个 epoch 的更新次数差别
for batch_size in (4, 10):
    batches = list(iterate_minibatches(features, batch_size))
    sizes = [len(b) for b in batches]
    print(f"batch_size={batch_size}: 一个 epoch 更新 {len(batches)} 次，各批大小 {sizes}")

# 注意：样本数不一定整除 batch size，最后一批会比较小，这是正常现象

## 优化器之争：SGD 对上 Adam

优化器（optimizer）决定「拿到梯度后怎么更新参数」。关键直觉一句话：更新就是让参数沿着负梯度 −∇L 的方向移动，学习率（learning rate）控制每一步走多大。

随机梯度下降（SGD, stochastic gradient descent）像用固定步幅走下坡；加上动量（momentum）会累积惯性、冲得更顺。Adam 则会依每个方向的历史梯度自动调整步幅（自适应学习率，adaptive learning rate），通常收敛更快更稳。学习率仍是最关键的旋钮。

| 优化器 | 步幅 | 特性 | 何时用 |
|---|---|---|---|
| SGD | 固定 | 单纯、需手调学习率 | 想精细控制、数据大时 |
| SGD + momentum | 固定 + 惯性 | 走得较顺、较快 | 想加速又保留 SGD 行为 |
| Adam | 自适应 | 收敛快、较不挑学习率 | 一开始的安全预设 |

In [ ]:
# 概念：在同一个自造线性回归任务上，分别用 SGD 与 Adam 训练，比较损失下降曲线
rng = np.random.default_rng(1)

# 造一份线性关系假数据：y = 2*x + 杂讯（x 当成标准化后的某个特征）
x = rng.normal(size=(60, 1))
true_w, true_b = 2.0, 0.5
y = true_w * x + true_b + 0.1 * rng.normal(size=(60, 1))

def loss_and_grad(w, b, x, y):
    pred = w * x + b
    error = pred - y
    mse = np.mean(error ** 2)            # 均方误差 mean squared error
    grad_w = np.mean(2 * error * x)      # 对 w 的梯度
    grad_b = np.mean(2 * error)          # 对 b 的梯度
    return mse, grad_w, grad_b

def train_sgd(lr=0.1, epochs=40):
    w, b = 0.0, 0.0
    history = []
    for _ in range(epochs):
        mse, gw, gb = loss_and_grad(w, b, x, y)
        w -= lr * gw                     # 沿负梯度方向移动固定步幅
        b -= lr * gb
        history.append(mse)
    return history

def train_adam(lr=0.1, epochs=40, beta1=0.9, beta2=0.999, eps=1e-8):
    w, b = 0.0, 0.0
    mw = mb = vw = vb = 0.0              # 一阶(动量)与二阶(梯度平方)的移动平均
    history = []
    for t in range(1, epochs + 1):
        mse, gw, gb = loss_and_grad(w, b, x, y)
        mw = beta1 * mw + (1 - beta1) * gw
        mb = beta1 * mb + (1 - beta1) * gb
        vw = beta2 * vw + (1 - beta2) * gw ** 2
        vb = beta2 * vb + (1 - beta2) * gb ** 2
        # 偏差校正 bias correction：补偿初期移动平均偏向 0 的问题
        mw_hat, mb_hat = mw / (1 - beta1 ** t), mb / (1 - beta1 ** t)
        vw_hat, vb_hat = vw / (1 - beta2 ** t), vb / (1 - beta2 ** t)
        w -= lr * mw_hat / (np.sqrt(vw_hat) + eps)   # 每个方向各自的自适应步幅
        b -= lr * mb_hat / (np.sqrt(vb_hat) + eps)
        history.append(mse)
    return history

sgd_hist = train_sgd()
adam_hist = train_adam()

plt.figure(figsize=(6, 4))
plt.plot(sgd_hist, label="SGD")
plt.plot(adam_hist, label="Adam")
plt.xlabel("epoch")
plt.ylabel("训练损失 MSE")
plt.title("SGD 与 Adam 的收敛比较")
plt.legend()
plt.show()

print("SGD 最终损失:", round(sgd_hist[-1], 5))
print("Adam 最终损失:", round(adam_hist[-1], 5))

## 切出验证集，看见过拟合

模型在训练集上变好，不等于它真的「学会」了。泛化（generalization）指的是对没见过的数据也表现好。要看泛化，就得把一部分数据藏起来当验证集（validation set）。

过拟合（overfitting）是模型把训练数据的杂讯都背了下来：训练损失持续下降，但验证损失反转上升，两者落差越拉越大。反过来，欠拟合（underfitting）是模型太弱，连训练集都学不好，两条损失都偏高。

形状：先打乱索引，依比例切出 train/validation 两份；训练只用 train，评估看 validation。

In [ ]:
# 概念：刻意把一个高弹性模型训练到过拟合，观察训练损失下降但验证损失反转上升
rng = np.random.default_rng(2)

# 造带杂讯的假资料：容积率 floor area ratio 对地价（单位简化）
far_raw = np.linspace(1.0, 5.0, 24).reshape(-1, 1)
land_price = 3.0 * far_raw + 1.2 * rng.normal(size=far_raw.shape) + 2.0   # 近似线性 + 较大杂讯

# 注意：高次方会让数值爆掉，先把容积率标准化到 [-1, 1] 附近，多项式特征才不会 overflow
far = (far_raw - far_raw.mean()) / (far_raw.max() - far_raw.min())

# 打乱后切成 train / validation（资料少，故意让训练集很小以放大过拟合）
idx = rng.permutation(len(far))
split = 8
train_idx, val_idx = idx[:split], idx[split:]

# 技巧：用高次多项式特征制造「过度弹性」的模型，最容易示范过拟合
def make_features(v, degree=9):
    return np.hstack([v ** d for d in range(degree + 1)])   # 1, v, v^2, ... 当作多个特征

X_train = make_features(far[train_idx])
X_val = make_features(far[val_idx])
y_train, y_val = land_price[train_idx], land_price[val_idx]

w = np.zeros((X_train.shape[1], 1))   # 每个多项式特征一个权重
lr, epochs = 0.5, 60000
train_curve, val_curve = [], []
for _ in range(epochs):
    error = X_train @ w - y_train
    grad = X_train.T @ error / len(X_train)   # 对权重的梯度
    w -= lr * grad
    train_curve.append(np.mean((X_train @ w - y_train) ** 2))
    val_curve.append(np.mean((X_val @ w - y_val) ** 2))   # 验证损失：没参与更新

plt.figure(figsize=(6, 4))
plt.plot(train_curve, label="训练损失")
plt.plot(val_curve, label="验证损失")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.title("过拟合：训练损失持续下降、验证损失反转")
plt.legend()
plt.show()

print("训练损失最终:", round(train_curve[-1], 4))
print("验证损失最低出现在 epoch:", int(np.argmin(val_curve)), "  最终验证损失:", round(val_curve[-1], 4))

## 正则化工具一：丢弃法 dropout

正则化（regularization）泛指「故意给模型一点限制」以换取更好的泛化。丢弃法（dropout）是其中最常用的一种。

做法：训练时每一步随机关掉一部分神经元（依丢弃率 dropout rate），逼网络不依赖单一路径、学到更稳健的特征。直觉上像考试前随机抽掉几页笔记练习，反而更会举一反三。关键注意点是训练与推论模式差异（train/eval mode）：推论（预测）时要关闭 dropout 并做缩放补偿，否则输出尺度会不对。

形状：训练时 输出 = 输入 * 随机遮罩 /(1 − rate)；推论时 输出 = 输入（不丢弃）。

In [ ]:
# 概念：手写 inverted dropout，示范训练模式会丢弃并缩放、推论模式原样输出
rng = np.random.default_rng(3)

# 造一层神经元的假输出（一批 5 个样本、每样本 6 个神经元）
activations = rng.uniform(1.0, 2.0, size=(5, 6))

def dropout(x, rate, training):
    if not training:
        return x                       # 注意：推论模式不丢弃，直接回传
    keep_prob = 1.0 - rate
    mask = (rng.uniform(size=x.shape) < keep_prob)   # 随机决定哪些神经元保留
    # 除以 keep_prob 做缩放，让训练与推论的期望输出一致
    return x * mask / keep_prob

train_out = dropout(activations, rate=0.5, training=True)
eval_out = dropout(activations, rate=0.5, training=False)

print("训练模式输出（部分被归零、其余放大）:\n", np.round(train_out, 2))
print("\n推论模式输出（与原输入相同）:\n", np.round(eval_out, 2))
# 期望值应接近：缩放让两种模式的整体平均对得起来
print("\n原输入平均:", round(activations.mean(), 3), " 训练输出平均:", round(train_out.mean(), 3))

In [ ]:
# 概念：在前面过拟合的多项式模型上，用权重衰减模拟正则化效果，比较验证损失的改善
rng = np.random.default_rng(2)   # 沿用同一份资料设定，重现过拟合场景

far_raw = np.linspace(1.0, 5.0, 24).reshape(-1, 1)
land_price = 3.0 * far_raw + 1.2 * rng.normal(size=far_raw.shape) + 2.0
far = (far_raw - far_raw.mean()) / (far_raw.max() - far_raw.min())   # 同样标准化避免 overflow
idx = rng.permutation(len(far))
split = 8
train_idx, val_idx = idx[:split], idx[split:]

def make_features(v, degree=9):
    return np.hstack([v ** d for d in range(degree + 1)])

X_train, X_val = make_features(far[train_idx]), make_features(far[val_idx])
y_train, y_val = land_price[train_idx], land_price[val_idx]

def train(reg_strength):
    w = np.zeros((X_train.shape[1], 1))
    val_curve = []
    for _ in range(60000):
        error = X_train @ w - y_train
        # 正则化项：对权重大小施加惩罚，逼模型别把杂讯背得太用力
        grad = X_train.T @ error / len(X_train) + reg_strength * w
        w -= 0.5 * grad
        val_curve.append(np.mean((X_val @ w - y_val) ** 2))
    return val_curve

no_reg = train(reg_strength=0.0)      # 不正则化（容易过拟合）
with_reg = train(reg_strength=0.005)  # 加入适度正则化
# 技巧：正则化强度要适中；太强反而会欠拟合，可扫几个值挑验证损失最低的

plt.figure(figsize=(6, 4))
plt.plot(no_reg, label="未加正则化")
plt.plot(with_reg, label="加入正则化")
plt.xlabel("epoch")
plt.ylabel("验证损失 MSE")
plt.title("正则化让验证损失反转幅度变小")
plt.legend()
plt.show()

print("未加正则化 最终验证损失:", round(no_reg[-1], 4), " 反转后最高:", round(max(no_reg[int(np.argmin(no_reg)):]), 4))
print("加入正则化 最终验证损失:", round(with_reg[-1], 4), " 反转后最高:", round(max(with_reg[int(np.argmin(with_reg)):]), 4))

## 正则化工具二：提早停止 early stopping

提早停止（early stopping）是说：与其硬训练固定世代数，不如盯着验证损失，一旦它连续好几个 epoch 不再进步就停手，并保留表现最好的那一版。

三个关键词：监控指针（monitor metric，通常是验证损失）、耐心值（patience，容忍几个 epoch 没进步）、最佳检查点（best checkpoint，把验证损失最低时的参数存起来，最后还原）。这样既省时间又自动避开过拟合区段。

形状：若验证损失创新低就更新检查点并归零计数；否则计数加一，计数超过 patience 就停。

In [ ]:
# 概念：自写 early stopping 监控回圈，记录并还原验证损失最低时的参数，标出停在第几个 epoch
rng = np.random.default_rng(2)

far_raw = np.linspace(1.0, 5.0, 24).reshape(-1, 1)
land_price = 3.0 * far_raw + 1.2 * rng.normal(size=far_raw.shape) + 2.0
far = (far_raw - far_raw.mean()) / (far_raw.max() - far_raw.min())   # 标准化避免高次方 overflow
idx = rng.permutation(len(far))
split = 8
train_idx, val_idx = idx[:split], idx[split:]

def make_features(v, degree=9):
    return np.hstack([v ** d for d in range(degree + 1)])

X_train, X_val = make_features(far[train_idx]), make_features(far[val_idx])
y_train, y_val = land_price[train_idx], land_price[val_idx]

w = np.zeros((X_train.shape[1], 1))
patience = 2000               # 容忍 2000 个 epoch 没进步
best_val = np.inf             # 目前看过的最低验证损失
best_w = w.copy()            # 最佳检查点
best_epoch = 0
wait = 0                     # 已连续几个 epoch 没进步
val_curve = []

for epoch in range(60000):
    error = X_train @ w - y_train
    w -= 0.5 * (X_train.T @ error / len(X_train))
    val_loss = np.mean((X_val @ w - y_val) ** 2)
    val_curve.append(val_loss)
    if val_loss < best_val - 1e-6:   # 注意：要有实质进步才算数，避免被微小浮动误导
        best_val, best_w, best_epoch = val_loss, w.copy(), epoch   # 创新低就存档
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            break             # 连续没进步达 patience，提早停止

w = best_w                   # 还原到最佳检查点
print("实际跑到 epoch:", epoch, "（总上限 60000）")
print("最佳检查点在 epoch:", best_epoch, "  最低验证损失:", round(best_val, 4))

plt.figure(figsize=(6, 4))
plt.plot(val_curve, label="验证损失")
plt.axvline(best_epoch, color="red", linestyle="--", label="最佳检查点")
plt.xlabel("epoch")
plt.ylabel("验证损失 MSE")
plt.title("提早停止：保留验证损失最低的版本")
plt.legend()
plt.show()

## 学习曲线判读与训练诊断

学习曲线（learning curve）是把训练损失与验证损失随 epoch 画出来的图，可以当成模型的体检报告。收敛（convergence）指损失趋于平缓、不再大幅下降。

看图决定动哪个旋钮：
- 两条都高且贴在一起：高偏差（high bias），也就是欠拟合，对策是加大模型、训练更久或加更好的特征。
- 训练低、验证高且落差大：高方差（high variance），也就是过拟合，对策是加正则化（dropout、提早停止）或拿更多数据。
- 两条都低且收敛：刚好，维持即可。

原则是先看图诊断，再决定下一步，不要盲目调参。

In [ ]:
# 概念：准备三组预造的损失曲线（欠拟合/刚好/过拟合），逐张对照判读并印出对策
epochs = np.arange(1, 61)

# 造三种典型曲线当示范（用简单函数仿真，不是真的训练出来的）
underfit = {
    "train": 1.5 - 0.3 * (1 - np.exp(-epochs / 20)),   # 训练损失也降不下去（停在高处）
    "val":   1.6 - 0.3 * (1 - np.exp(-epochs / 20)),
}
good_fit = {
    "train": 1.5 * np.exp(-epochs / 12) + 0.1,         # 两条都降到低位且贴近
    "val":   1.5 * np.exp(-epochs / 12) + 0.15,
}
overfit = {
    "train": 1.5 * np.exp(-epochs / 8) + 0.05,         # 训练持续降
    "val":   1.5 * np.exp(-epochs / 8) + 0.05 + 0.02 * epochs,   # 验证反转上升
}

def diagnose(train_curve, val_curve):
    final_train, final_val = train_curve[-1], val_curve[-1]
    gap = final_val - final_train
    if final_train > 0.8:                      # 连训练都学不好
        return "高偏差（欠拟合）", "加大模型 / 训练更久 / 加更好的特征"
    if gap > 0.3:                              # 训练好但验证差很多
        return "高方差（过拟合）", "加正则化（dropout / early stopping）或拿更多数据"
    return "刚好（收敛良好）", "维持现状即可"

cases = [("欠拟合", underfit), ("刚好", good_fit), ("过拟合", overfit)]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, curves) in zip(axes, cases):
    ax.plot(epochs, curves["train"], label="训练")
    ax.plot(epochs, curves["val"], label="验证")
    ax.set_title(name)
    ax.set_xlabel("epoch")
    ax.set_ylabel("损失")
    ax.legend()
plt.tight_layout()
plt.show()

for name, curves in cases:
    label, advice = diagnose(curves["train"], curves["val"])
    print(f"{name} -> 判读: {label}; 对策: {advice}")

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）为楼高回归挑一个优化器（集成：小批量 + SGD/Adam）
#   情境：用自造的「楼层数对楼高」数据做一个最小回归，想知道哪个优化器收敛比较快。
#   要求：
#     1. 用 numpy 自造数据（楼层数对应楼高，约略线性 + 一点杂讯），切成固定 batch size 的小批量。
#     2. 同样的网络分别用 SGD 与 Adam 各训练数个 epoch，记录每个 epoch 的训练损失。
#     3. 把两条损失曲线画在同一张图上。
#   验收：应该看到两条下降曲线，Adam 通常更早趋于平缓。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）抓出过拟合并用正则化救回（集成：验证切分 + 过拟合 + dropout/正则化 + 学习曲线）
#   情境：用自造的「容积率与面积对地价」数据训练一个容易过拟合的小网络。
#   要求：
#     1. 做 train/validation split，先训练到验证损失明显反转上升。
#     2. 在训练流程中加入正则化（dropout 遮罩或权重衰减皆可）后重新训练。
#     3. 把加入前后的训练与验证损失曲线并排比较。
#   验收：应该看到加入正则化后，验证损失反转的幅度变小、训练与验证的落差缩小。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）自制训练诊断器（集成：early stopping + 学习曲线 + 优化器 + 对策决策）
#   情境：把都市分区自造数据的训练流程包成一个可重复实验的小工具，让它能自动停下并给诊断建议。
#   要求：
#     1. 写一个训练循环，内置 early stopping（含 patience 与最佳检查点还原）。
#     2. 跑两种设置（例如不同学习率，或有无正则化），各自输出学习曲线。
#     3. 依曲线形状以简单规则自动判断属于欠拟合或过拟合，并印出建议的下一步对策。
#   验收：应看到工具在验证损失停滞时自动停止，并针对每次实验印出「高偏差／高方差」判读与一句对策建议。
# TODO: 在这里完成


## 小结

- 小批量（mini-batch）让更新更频繁、内存更省，梯度杂讯还有助于跳出坏点；DataLoader 是自动分批与洗牌的搬运工，一个 epoch 的更新次数约为 样本数 / batch size。
- 优化器决定怎么用梯度更新参数：SGD 用固定步幅、Adam 自适应每个方向的步幅且通常收敛更快更稳；学习率始终是最关键的旋钮。
- 切出验证集才看得到泛化能力；训练损失降、验证损失反转上升就是过拟合的信号，两条都高则是欠拟合。
- 丢弃法（dropout）训练时随机关神经元、推论时关闭并缩放补偿；它和权重衰减一样属于正则化，能缩小训练与验证的落差。
- 提早停止（early stopping）盯着验证损失，连续 patience 个 epoch 没进步就停，并还原最佳检查点，省时又避免过拟合。
- 学习曲线是模型的体检报告：先看图判断高偏差或高方差，再决定加大模型、加正则化或补数据，不要盲目调参。